# FiQA BGE-small RARS Transfer

FiQA transfer notebook for RARS cross-setting validation.

This version builds a FiQA-specific frozen IVF-PQ index and candidate cache, rather than reusing external candidate caches.

Main fixes:

1. Use `heldout_eval_split.json` query rows to select the correct 1,000 evaluation queries.
2. Rebuild Top-100 ANN candidates from the current FiQA frozen IVF-PQ index.
3. Store candidate ids as **internal row ids**, so they can correctly index residuals and sidecar codes.
4. Recompute exact candidate scores against the current FiQA embeddings.
5. Re-run PCA / score-error weighted / boundary-weighted basis diagnostics using aligned candidates.

The first goal is not final Recall@10 yet. The first goal is to verify whether a retrieval-aware basis improves the candidate-level exact-score proxy over the existing PCA basis.

In [ ]:
from google.colab import drive

drive.mount("/content/gdrive", force_remount=True)

## 1. Imports, paths, and global configuration

In [ ]:
from pathlib import Path
import json
import os
import gc
import time
import math
import hashlib
import csv
import zipfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    !pip -q install torch
    import torch

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    !pip -q install "sentence-transformers==3.4.1"
    from sentence_transformers import SentenceTransformer

try:
    import faiss
except Exception:
    !pip -q install faiss-gpu-cu12 || pip -q install faiss-gpu || pip -q install faiss-cpu
    import faiss

ROOT = Path("/content/gdrive/MyDrive/rag-pq-checkpoints")

SETTING_SLUG = "fiqa_bge_small"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

DATA_DIR = ROOT / "beir_data"
FIQA_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
FIQA_DIR = DATA_DIR / "fiqa"

CACHE_DIR = ROOT / f"{SETTING_SLUG}_rars_transfer_cache"
OUT_DIR = ROOT / f"retrieval_aware_residual_basis_{SETTING_SLUG}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

DOC_EMB_PATH = CACHE_DIR / "embeddings.fp16.memmap"
DOC_IDS_PATH = CACHE_DIR / "doc_ids.int64.memmap"
DOC_ID_STRINGS_PATH = CACHE_DIR / "doc_id_strings.json"
QUERY_PATH = CACHE_DIR / "query_vectors.fp32.npy"
EVAL_QIDS_PATH = CACHE_DIR / "eval_qids.json"
QRELS_PATH = CACHE_DIR / "qrels.json"
INDEX_PATH = CACHE_DIR / "frozen_ivfpq_m32_nlist256.index"
CACHE_META_PATH = CACHE_DIR / "metadata.json"

DIM = 384
TOP_L = 100
TOP_B = 40
FINAL_K = 10
RANK = 16
NPROBE = 16
NLIST = 256
PQ_M = 32
PQ_NBITS = 8
DOC_EMBED_BATCH_SIZE = 256
QUERY_EMBED_BATCH_SIZE = 256
ALPHA_DEFAULT = 1.0

CONFIG = {
    "dataset": "FiQA / BEIR transfer split",
    "embedding_model": EMBEDDING_MODEL,
    "dimension": DIM,
    "frozen_index": "IVF-PQ M=32 nlist=256 nprobe=16",
    "candidate_top_l": TOP_L,
    "correction_top_b": TOP_B,
    "final_k": FINAL_K,
    "sidecar_rank": RANK,
    "note": "FiQA RARS transfer notebook. It builds its own FiQA embeddings, frozen IVF-PQ index, and Top100 candidate cache.",
}

with open(OUT_DIR / "config_fixed.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("setting:", SETTING_SLUG)
print("model:", EMBEDDING_MODEL)
print("out_dir:", OUT_DIR)
print("cache_dir:", CACHE_DIR)
print("faiss version:", faiss.__version__)


## 2. Load FiQA, encode embeddings, and build the frozen IVF-PQ index

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def download_and_extract_fiqa():
    DATA_DIR.mkdir(exist_ok=True, parents=True)
    zip_path = DATA_DIR / "fiqa.zip"

    if not FIQA_DIR.exists():
        if not zip_path.exists():
            print("Downloading FiQA archive...")
            urllib.request.urlretrieve(FIQA_URL, zip_path)
        print("Extracting FiQA archive...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DATA_DIR)

    required = [
        FIQA_DIR / "corpus.jsonl",
        FIQA_DIR / "queries.jsonl",
        FIQA_DIR / "qrels" / "test.tsv",
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"FiQA extraction incomplete. Missing: {missing}")


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def read_qrels(path):
    qrels = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            qid = str(row["query-id"])
            docid = str(row["corpus-id"])
            score = int(row["score"])
            if score > 0:
                qrels.setdefault(qid, {})[docid] = score
    return qrels


def encode_texts(texts, model_name, batch_size):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=device,
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return vectors.astype(np.float32)


def write_fp16_memmap(path, arr):
    mm = np.memmap(path, dtype=np.float16, mode="w+", shape=arr.shape)
    mm[:] = arr.astype(np.float16)
    mm.flush()
    return mm


def write_int64_memmap(path, arr):
    arr = np.asarray(arr, dtype=np.int64)
    mm = np.memmap(path, dtype=np.int64, mode="w+", shape=arr.shape)
    mm[:] = arr
    mm.flush()
    return mm


download_and_extract_fiqa()

corpus_rows = read_jsonl(FIQA_DIR / "corpus.jsonl")
query_rows_raw = read_jsonl(FIQA_DIR / "queries.jsonl")
all_qrels = read_qrels(FIQA_DIR / "qrels" / "test.tsv")

doc_id_strings = [str(row["_id"]) for row in corpus_rows]
doc_texts = [
    ((row.get("title") or "") + "\n" + (row.get("text") or "")).strip()
    for row in corpus_rows
]
doc_id_to_index = {doc_id: i for i, doc_id in enumerate(doc_id_strings)}

query_ids, query_texts, qrels = [], [], {}
for row in query_rows_raw:
    qid = str(row["_id"])
    if qid not in all_qrels:
        continue

    filtered = {
        str(docid): int(score)
        for docid, score in all_qrels[qid].items()
        if str(docid) in doc_id_to_index and int(score) > 0
    }
    if not filtered:
        continue

    query_ids.append(qid)
    query_texts.append(row.get("text") or "")
    qrels[qid] = filtered

N_DOCS = len(doc_texts)
N_QUERIES = len(query_texts)

print("FiQA documents:", N_DOCS)
print("FiQA evaluated queries:", N_QUERIES)
print("Example doc id:", doc_id_strings[0])
print("Example qid:", query_ids[0])

try:
    doc_ids_int = np.asarray([int(x) for x in doc_id_strings], dtype=np.int64)
except ValueError:
    print("Non-numeric doc ids detected; using internal row ids as doc_ids.")
    doc_ids_int = np.arange(N_DOCS, dtype=np.int64)

if DOC_EMB_PATH.exists() and DOC_IDS_PATH.exists() and QUERY_PATH.exists():
    X = np.memmap(DOC_EMB_PATH, dtype=np.float16, mode="r", shape=(N_DOCS, DIM))
    doc_ids = np.memmap(DOC_IDS_PATH, dtype=np.int64, mode="r", shape=(N_DOCS,))
    Q = np.load(QUERY_PATH).astype(np.float32)
    print("Loaded cached FiQA embeddings.")
else:
    print("Encoding FiQA documents...")
    X_docs = encode_texts(doc_texts, EMBEDDING_MODEL, DOC_EMBED_BATCH_SIZE)
    print("Encoding FiQA queries...")
    X_queries = encode_texts(query_texts, EMBEDDING_MODEL, QUERY_EMBED_BATCH_SIZE)

    assert X_docs.shape[1] == DIM, X_docs.shape
    assert X_queries.shape[1] == DIM, X_queries.shape

    X = write_fp16_memmap(DOC_EMB_PATH, X_docs)
    doc_ids = write_int64_memmap(DOC_IDS_PATH, doc_ids_int)
    np.save(QUERY_PATH, X_queries.astype(np.float32))
    Q = X_queries.astype(np.float32)

    del X_docs, X_queries
    gc.collect()

save_json(doc_id_strings, DOC_ID_STRINGS_PATH)
save_json(query_ids, EVAL_QIDS_PATH)
save_json(qrels, QRELS_PATH)

eval_qids = query_ids
eval_query_rows = np.arange(len(query_ids), dtype=np.int64)
eval_query_ids = [str(x) for x in query_ids]

print("X:", X.shape, X.dtype)
print("doc_ids:", doc_ids.shape, doc_ids.dtype)
print("Q:", Q.shape, Q.dtype)
print("eval_query_rows:", eval_query_rows.shape)


def build_or_load_frozen_ivfpq_index():
    if INDEX_PATH.exists():
        idx = faiss.read_index(str(INDEX_PATH))
        idx.nprobe = NPROBE
        try:
            idx.make_direct_map()
        except Exception as e:
            print("make_direct_map warning:", repr(e))
        print("Loaded frozen IVF-PQ index:", INDEX_PATH)
        return idx

    print("Building frozen IVF-PQ index...")
    quantizer = faiss.IndexFlatIP(DIM)
    idx = faiss.IndexIVFPQ(
        quantizer,
        DIM,
        NLIST,
        PQ_M,
        PQ_NBITS,
        faiss.METRIC_INNER_PRODUCT,
    )

    train_x = np.asarray(X, dtype=np.float32)
    idx.train(train_x)
    idx.add(train_x)
    idx.nprobe = NPROBE

    try:
        idx.make_direct_map()
    except Exception as e:
        print("make_direct_map warning:", repr(e))

    faiss.write_index(idx, str(INDEX_PATH))
    print("Saved frozen IVF-PQ index:", INDEX_PATH)
    return idx


index = build_or_load_frozen_ivfpq_index()

metadata = {
    "dataset": "FiQA / BEIR",
    "num_documents": int(N_DOCS),
    "num_queries": int(N_QUERIES),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": int(DIM),
    "index": {
        "type": "IndexIVFPQ",
        "nlist": int(NLIST),
        "M": int(PQ_M),
        "nbits": int(PQ_NBITS),
        "metric": "inner_product",
        "nprobe": int(NPROBE),
    },
}
save_json(metadata, CACHE_META_PATH)

print("index ntotal:", index.ntotal, "d:", index.d, "nprobe:", index.nprobe)
assert index.ntotal == N_DOCS
assert index.d == DIM


## 3. Build current FiQA candidate cache

The candidate cache is generated from the current FiQA IVF-PQ index and uses internal row ids.

In [ ]:
CURRENT_CACHE_DIR = OUT_DIR / "current_fiqa_candidate_cache"
CURRENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

CURR_ANN_ROWS_PATH = CURRENT_CACHE_DIR / "ivfpq_m32_nprobe16_top100_internal_rows.npy"
CURR_ANN_SCORES_PATH = CURRENT_CACHE_DIR / "ivfpq_m32_nprobe16_top100_scores.npy"
CURR_EXACT_SCORES_PATH = CURRENT_CACHE_DIR / "candidate_exact_scores.npy"
CURR_IS_EXACT_TOP10_PATH = CURRENT_CACHE_DIR / "candidate_is_exact_top10.npy"
CURR_CUTOFF_MARGIN_PATH = CURRENT_CACHE_DIR / "candidate_cutoff_margin.npy"
CURR_META_PATH = CURRENT_CACHE_DIR / "metadata.json"


def build_current_ann_cache(Q, index, eval_query_rows, top_l=100, batch_size=64):
    all_scores = []
    all_rows = []

    for start in range(0, len(eval_query_rows), batch_size):
        end = min(start + batch_size, len(eval_query_rows))
        qidx = eval_query_rows[start:end]
        q_batch = Q[qidx].astype(np.float32)

        scores, rows = index.search(q_batch, top_l)

        all_scores.append(scores.astype(np.float32))
        all_rows.append(rows.astype(np.int64))

        print(f"searched {end}/{len(eval_query_rows)}")

    return np.vstack(all_scores), np.vstack(all_rows)


if CURR_ANN_ROWS_PATH.exists() and CURR_ANN_SCORES_PATH.exists():
    ann_internal_rows = np.load(CURR_ANN_ROWS_PATH).astype(np.int64)
    ann_scores_curr = np.load(CURR_ANN_SCORES_PATH).astype(np.float32)
    print("loaded current ANN cache")
else:
    ann_scores_curr, ann_internal_rows = build_current_ann_cache(
        Q=Q,
        index=index,
        eval_query_rows=eval_query_rows,
        top_l=TOP_L,
        batch_size=64,
    )
    np.save(CURR_ANN_ROWS_PATH, ann_internal_rows)
    np.save(CURR_ANN_SCORES_PATH, ann_scores_curr)
    print("saved current ANN cache")

metadata = {
    "top_l": TOP_L,
    "nprobe": NPROBE,
    "num_queries": int(len(eval_query_rows)),
    "index_path": str(INDEX_PATH),
    "candidate_id_space": "internal row ids for current FiQA IVF-PQ index",
}
with open(CURR_META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("ann_internal_rows:", ann_internal_rows.shape, ann_internal_rows.dtype)
print("ann_scores_curr:", ann_scores_curr.shape, ann_scores_curr.dtype)
print("top5 internal row ids:", ann_internal_rows[0, :5])
print("top5 FiQA doc ids:", np.asarray(doc_ids[ann_internal_rows[0, :5]], dtype=np.int64))


## 4. Recompute exact scores for the current candidate pool

In [ ]:
def build_current_exact_candidate_scores(Q, X, eval_query_rows, ann_internal_rows, batch_size=64):
    n_q, top_l = ann_internal_rows.shape
    exact_scores = np.empty((n_q, top_l), dtype=np.float32)

    for start in range(0, n_q, batch_size):
        end = min(start + batch_size, n_q)

        for qi in range(start, end):
            qrow = int(eval_query_rows[qi])
            q = Q[qrow].astype(np.float32)
            ids = ann_internal_rows[qi]
            xb = np.asarray(X[ids], dtype=np.float32)
            exact_scores[qi] = xb @ q

        print(f"exact scored {end}/{n_q}")

    return exact_scores


if CURR_EXACT_SCORES_PATH.exists():
    exact_scores_curr = np.load(CURR_EXACT_SCORES_PATH).astype(np.float32)
    print("loaded current exact scores")
else:
    exact_scores_curr = build_current_exact_candidate_scores(
        Q=Q,
        X=X,
        eval_query_rows=eval_query_rows,
        ann_internal_rows=ann_internal_rows,
        batch_size=64,
    )
    np.save(CURR_EXACT_SCORES_PATH, exact_scores_curr)
    print("saved current exact scores")

print("exact_scores_curr:", exact_scores_curr.shape, exact_scores_curr.dtype)
print("ANN score range:", float(ann_scores_curr.min()), float(ann_scores_curr.max()))
print("exact score range:", float(exact_scores_curr.min()), float(exact_scores_curr.max()))
print("candidate score MSE:", float(np.mean((ann_scores_curr - exact_scores_curr) ** 2)))

## 5. Exact-candidate Top-10 labels and cutoff margins

In [ ]:
exact_order = np.argsort(-exact_scores_curr, axis=1)

is_exact_top10_curr = np.zeros_like(exact_scores_curr, dtype=bool)
for i in range(exact_scores_curr.shape[0]):
    is_exact_top10_curr[i, exact_order[i, :FINAL_K]] = True

cutoff_score = np.take_along_axis(
    exact_scores_curr,
    exact_order[:, [FINAL_K - 1]],
    axis=1,
)
cutoff_margin_curr = exact_scores_curr - cutoff_score

np.save(CURR_IS_EXACT_TOP10_PATH, is_exact_top10_curr)
np.save(CURR_CUTOFF_MARGIN_PATH, cutoff_margin_curr)

base_order = np.argsort(-ann_scores_curr, axis=1)
base_overlaps = []
for i in range(exact_scores_curr.shape[0]):
    exact_top10 = set(exact_order[i, :FINAL_K].tolist())
    base_top10 = set(base_order[i, :FINAL_K].tolist())
    base_overlaps.append(len(exact_top10 & base_top10) / FINAL_K)

print("is_exact_top10_curr:", is_exact_top10_curr.shape)
print("cutoff_margin_curr:", cutoff_margin_curr.shape)
print("base exact-candidate top10 overlap:", float(np.mean(base_overlaps)))

## 6. Build aligned frozen IVF-PQ residual memmap

This reconstructs vectors from the current frozen index and computes `R = X - xhat_PQ`.

In [ ]:
RESIDUAL_PATH = OUT_DIR / "residual_ivfpq_m32.float32.memmap"


def build_residual_memmap(index, X, out_path, n_docs=N_DOCS, dim=DIM, batch_size=50_000):
    R = np.memmap(out_path, dtype=np.float32, mode="w+", shape=(n_docs, dim))

    for start in range(0, n_docs, batch_size):
        end = min(start + batch_size, n_docs)
        ids = np.arange(start, end, dtype=np.int64)

        xhat = index.reconstruct_batch(ids).astype(np.float32)
        xb = np.asarray(X[start:end], dtype=np.float32)
        R[start:end] = xb - xhat
        R.flush()

        print(f"residual {end}/{n_docs}")

    return R


if RESIDUAL_PATH.exists():
    R = np.memmap(RESIDUAL_PATH, dtype=np.float32, mode="r", shape=(N_DOCS, DIM))
    print("loaded residual memmap:", R.shape, R.dtype)
else:
    R = build_residual_memmap(index, X, RESIDUAL_PATH)
    print("built residual memmap:", R.shape, R.dtype)

## 7. Train retrieval-aware weighted residual bases

These are simple weighted-SVD baselines. They are intentionally not neural losses yet.

Compared bases:

- `pca_current`: existing Gate3 PCA residual basis.
- `score_error_weighted`: candidate documents weighted by absolute score error.
- `top10_boundary_weighted`: candidate documents weighted by exact Top-10 membership and closeness to the candidate Top-10 cutoff.

In [ ]:
def train_weighted_basis_from_candidate_docs(
    R,
    ann_internal_rows,
    weights_2d,
    rank=16,
    max_samples=300_000,
    seed=42,
    out_path=None,
):
    rng = np.random.default_rng(seed)

    flat_docs = ann_internal_rows.reshape(-1).astype(np.int64)
    flat_weights = weights_2d.reshape(-1).astype(np.float64)

    valid = (flat_docs >= 0) & np.isfinite(flat_weights) & (flat_weights > 0)
    flat_docs = flat_docs[valid]
    flat_weights = flat_weights[valid]

    # Combine repeated candidate appearances per document.
    df = pd.DataFrame({"doc": flat_docs, "w": flat_weights})
    agg = df.groupby("doc", sort=False)["w"].sum()
    docs = agg.index.to_numpy(dtype=np.int64)
    weights = agg.to_numpy(dtype=np.float64)
    weights = weights / (weights.sum() + 1e-12)

    take = min(max_samples, len(docs))
    if take < len(docs):
        sampled_docs = rng.choice(docs, size=take, replace=False, p=weights)
    else:
        sampled_docs = docs

    # Use document weights for weighted SVD.
    weight_lookup = dict(zip(docs.tolist(), weights.tolist()))
    sampled_w = np.array([weight_lookup[int(d)] for d in sampled_docs], dtype=np.float32)
    sampled_w = sampled_w / (sampled_w.mean() + 1e-12)

    R_sample = np.asarray(R[sampled_docs], dtype=np.float32)
    Rw = R_sample * np.sqrt(sampled_w[:, None])

    print("weighted SVD sample:", Rw.shape)
    _, _, vh = np.linalg.svd(Rw, full_matrices=False)
    B = vh[:rank].T.astype(np.float32)

    ortho_err = float(np.linalg.norm(B.T @ B - np.eye(rank)))
    print("orthogonality error:", ortho_err)

    if out_path is not None:
        np.save(out_path, B)
        print("saved:", out_path)

    return B


# Current-setting PCA baseline trained from FiQA residuals.
def train_pca_basis_from_residuals(R, rank=RANK, max_samples=300_000, seed=42):
    rng = np.random.default_rng(seed)
    n = R.shape[0]
    take = min(max_samples, n)
    rows = rng.choice(n, size=take, replace=False)

    Xr = np.asarray(R[rows], dtype=np.float32)
    Xr = Xr - Xr.mean(axis=0, keepdims=True)

    print("training current PCA residual basis:", Xr.shape)
    _, _, vt = np.linalg.svd(Xr, full_matrices=False)
    B = vt[:rank].T.astype(np.float32)

    # Orthonormal sanity check.
    gram = B.T @ B
    print("PCA basis:", B.shape, "orth_err:", float(np.max(np.abs(gram - np.eye(rank)))))
    return B


B_pca = train_pca_basis_from_residuals(R, rank=RANK)
np.save(OUT_DIR / "basis_pca_current_rank16.npy", B_pca)


# Weight 1: absolute candidate score error.
score_error_curr = exact_scores_curr - ann_scores_curr
w_error = np.abs(score_error_curr)
w_error = w_error / (np.mean(w_error) + 1e-12)
w_error = 1.0 + 5.0 * w_error

# Weight 2: exact Top-10 + near-boundary emphasis.
margin_abs = np.abs(cutoff_margin_curr)
margin_weight = 1.0 / (margin_abs + 1e-3)
margin_weight = margin_weight / (np.mean(margin_weight) + 1e-12)

w_boundary = (
    1.0
    + 8.0 * is_exact_top10_curr.astype(np.float32)
    + 4.0 * margin_weight.astype(np.float32)
)

basis_paths = {
    "pca_current": OUT_DIR / "basis_pca_current_rank16.npy",
    "score_error_weighted": OUT_DIR / "basis_score_error_weighted_rank16_internal.npy",
    "top10_boundary_weighted": OUT_DIR / "basis_top10_boundary_weighted_rank16_internal.npy",
}

if basis_paths["score_error_weighted"].exists():
    B_error = np.load(basis_paths["score_error_weighted"]).astype(np.float32)
    print("loaded score_error_weighted basis")
else:
    B_error = train_weighted_basis_from_candidate_docs(
        R=R,
        ann_internal_rows=ann_internal_rows,
        weights_2d=w_error,
        rank=RANK,
        max_samples=300_000,
        out_path=basis_paths["score_error_weighted"],
    )

if basis_paths["top10_boundary_weighted"].exists():
    B_boundary = np.load(basis_paths["top10_boundary_weighted"]).astype(np.float32)
    print("loaded top10_boundary_weighted basis")
else:
    B_boundary = train_weighted_basis_from_candidate_docs(
        R=R,
        ann_internal_rows=ann_internal_rows,
        weights_2d=w_boundary,
        rank=RANK,
        max_samples=300_000,
        out_path=basis_paths["top10_boundary_weighted"],
    )

basis_map = {
    "pca_current": B_pca,
    "score_error_weighted": B_error,
    "top10_boundary_weighted": B_boundary,
}

print("basis ready:", {k: v.shape for k, v in basis_map.items()})

## 8. Build int8 sidecar coefficients for each basis

In [ ]:
def build_int8_codes_for_basis(R, B, name, batch_size=100_000):
    code_path = OUT_DIR / f"codes_{name}_rank16.int8.memmap"
    scale_path = OUT_DIR / f"scales_{name}_rank16.float32.npy"

    if code_path.exists() and scale_path.exists():
        codes = np.memmap(code_path, dtype=np.int8, mode="r", shape=(N_DOCS, RANK))
        scales = np.load(scale_path).astype(np.float32)
        print("loaded codes:", name)
        return codes, scales

    # Pass 1: coefficient max abs for symmetric per-dimension int8 quantization.
    max_abs = np.zeros(RANK, dtype=np.float32)

    for start in range(0, N_DOCS, batch_size):
        end = min(start + batch_size, N_DOCS)
        coeff = np.asarray(R[start:end], dtype=np.float32) @ B
        max_abs = np.maximum(max_abs, np.max(np.abs(coeff), axis=0))
        print(f"{name} maxabs {end}/{N_DOCS}")

    scales = (max_abs + 1e-12) / 127.0
    np.save(scale_path, scales.astype(np.float32))

    # Pass 2: quantize and write codes.
    codes = np.memmap(code_path, dtype=np.int8, mode="w+", shape=(N_DOCS, RANK))

    for start in range(0, N_DOCS, batch_size):
        end = min(start + batch_size, N_DOCS)
        coeff = np.asarray(R[start:end], dtype=np.float32) @ B
        q = np.clip(np.round(coeff / scales[None, :]), -127, 127).astype(np.int8)
        codes[start:end] = q
        codes.flush()
        print(f"{name} codes {end}/{N_DOCS}")

    return codes, scales.astype(np.float32)


sidecars = {}
for name, B in basis_map.items():
    sidecars[name] = build_int8_codes_for_basis(R, B, name)

print("sidecars ready:", list(sidecars.keys()))

## 9. Candidate-level proxy diagnostics

This measures whether sidecar correction moves ANN candidate scores closer to exact inner-product scores within the same candidate pool.

This is not final qrels Recall@10. It is a useful diagnostic for whether a basis is aligned with recoverable candidate ranking error.

In [ ]:
def get_eval_query(row):
    return Q[int(eval_query_rows[row])].astype(np.float32)


def compute_correction_matrix(B, codes, scales, top_b=TOP_B):
    corr_mat = np.zeros_like(ann_scores_curr, dtype=np.float32)

    for qi in range(ann_internal_rows.shape[0]):
        q = get_eval_query(qi)
        q_proj = q @ B

        ids = ann_internal_rows[qi, :top_b]
        coeff = codes[ids].astype(np.float32) * scales[None, :]
        correction = coeff @ q_proj

        corr_mat[qi, :top_b] = correction.astype(np.float32)

    return corr_mat


def correction_error_correlation_current(name, B, codes, scales, top_b=TOP_B):
    corr_mat = compute_correction_matrix(B, codes, scales, top_b=top_b)
    target = exact_scores_curr - ann_scores_curr

    c = corr_mat[:, :top_b].reshape(-1)
    e = target[:, :top_b].reshape(-1)

    pearson = np.corrcoef(c, e)[0, 1]
    sign_agreement = np.mean(np.sign(c) == np.sign(e))
    best_alpha = float(np.dot(c, e) / (np.dot(c, c) + 1e-12))

    return {
        "basis": name,
        "pearson_corr_with_exact_minus_ann": float(pearson),
        "sign_agreement": float(sign_agreement),
        "best_alpha_mse": best_alpha,
        "correction_mean": float(c.mean()),
        "correction_std": float(c.std()),
        "target_mean": float(e.mean()),
        "target_std": float(e.std()),
    }


def evaluate_candidate_score_proxy_alpha_current(name, B, codes, scales, alphas, top_b=TOP_B):
    rows = []
    corr_mat = compute_correction_matrix(B, codes, scales, top_b=top_b)

    base_mse = np.mean((ann_scores_curr - exact_scores_curr) ** 2)

    exact_order_local = np.argsort(-exact_scores_curr, axis=1)
    exact_top10_set = [set(exact_order_local[i, :FINAL_K].tolist()) for i in range(exact_order_local.shape[0])]

    base_order = np.argsort(-ann_scores_curr, axis=1)
    base_overlap = []
    for i in range(ann_internal_rows.shape[0]):
        base_top10 = set(base_order[i, :FINAL_K].tolist())
        base_overlap.append(len(base_top10 & exact_top10_set[i]) / FINAL_K)
    base_overlap = float(np.mean(base_overlap))

    for alpha in alphas:
        corrected = ann_scores_curr + alpha * corr_mat
        corr_mse = np.mean((corrected - exact_scores_curr) ** 2)

        corr_order = np.argsort(-corrected, axis=1)
        corr_overlap = []

        for i in range(ann_internal_rows.shape[0]):
            corr_top10 = set(corr_order[i, :FINAL_K].tolist())
            corr_overlap.append(len(corr_top10 & exact_top10_set[i]) / FINAL_K)

        rows.append({
            "basis": name,
            "alpha": float(alpha),
            "base_mse": float(base_mse),
            "corrected_mse": float(corr_mse),
            "mse_reduction_pct": float((base_mse - corr_mse) / base_mse * 100),
            "base_top10_overlap": base_overlap,
            "corrected_top10_overlap": float(np.mean(corr_overlap)),
            "overlap_gain": float(np.mean(corr_overlap) - base_overlap),
        })

    return rows

## 10. Run diagnostics for all bases

In [ ]:
diag_rows = []
for name, B in basis_map.items():
    codes, scales = sidecars[name]
    diag_rows.append(
        correction_error_correlation_current(
            name=name,
            B=B,
            codes=codes,
            scales=scales,
            top_b=TOP_B,
        )
    )

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(OUT_DIR / "basis_correction_error_diagnostic_current.csv", index=False)
diag_df

In [ ]:
alphas = np.array([
    -2.0, -1.5, -1.0, -0.75, -0.5, -0.25, -0.1,
     0.0,
     0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0,
], dtype=np.float32)

alpha_rows = []
for name, B in basis_map.items():
    codes, scales = sidecars[name]
    alpha_rows.extend(
        evaluate_candidate_score_proxy_alpha_current(
            name=name,
            B=B,
            codes=codes,
            scales=scales,
            alphas=alphas,
            top_b=TOP_B,
        )
    )

alpha_df = pd.DataFrame(alpha_rows)
alpha_df.to_csv(OUT_DIR / "basis_alpha_sweep_proxy_current.csv", index=False)

alpha_df.sort_values(
    ["corrected_top10_overlap", "mse_reduction_pct"],
    ascending=False,
).head(30)

## 11. Summarize best alpha per basis

In [ ]:
best_by_overlap = (
    alpha_df.sort_values(["basis", "corrected_top10_overlap", "mse_reduction_pct"], ascending=[True, False, False])
    .groupby("basis")
    .head(1)
    .reset_index(drop=True)
)

best_by_mse = (
    alpha_df.sort_values(["basis", "mse_reduction_pct", "corrected_top10_overlap"], ascending=[True, False, False])
    .groupby("basis")
    .head(1)
    .reset_index(drop=True)
)

best_summary = best_by_overlap.merge(
    best_by_mse,
    on="basis",
    suffixes=("_best_overlap", "_best_mse"),
)

best_summary.to_csv(OUT_DIR / "basis_proxy_best_summary_current.csv", index=False)
best_summary

## 12. Optional: Top-B ablation for the best basis

Run this after checking the table above. It shows whether correction depth 10/20/40/100 changes proxy quality.

In [ ]:
# Pick the best basis by overlap proxy.
best_basis_name = best_by_overlap.sort_values("corrected_top10_overlap", ascending=False).iloc[0]["basis"]
best_alpha = float(best_by_overlap.sort_values("corrected_top10_overlap", ascending=False).iloc[0]["alpha"])

print("best basis by overlap:", best_basis_name)
print("best alpha:", best_alpha)

B_best = basis_map[best_basis_name]
codes_best, scales_best = sidecars[best_basis_name]

topb_rows = []
for tb in [0, 10, 20, 40, 100]:
    if tb == 0:
        rows = evaluate_candidate_score_proxy_alpha_current(
            name=f"{best_basis_name}_top0",
            B=B_best,
            codes=codes_best,
            scales=scales_best,
            alphas=np.array([0.0], dtype=np.float32),
            top_b=TOP_B,
        )
        row = rows[0]
        row["actual_top_b"] = 0
    else:
        rows = evaluate_candidate_score_proxy_alpha_current(
            name=f"{best_basis_name}_top{tb}",
            B=B_best,
            codes=codes_best,
            scales=scales_best,
            alphas=np.array([best_alpha], dtype=np.float32),
            top_b=tb,
        )
        row = rows[0]
        row["actual_top_b"] = tb
    topb_rows.append(row)

topb_df = pd.DataFrame(topb_rows)
topb_df.to_csv(OUT_DIR / "best_basis_topb_proxy_ablation_current.csv", index=False)
topb_df

## 13. Save manifest

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "package": "retrieval_aware_residual_basis_fiqa_bge_small",
    "note": "FiQA RARS transfer outputs; aligned internal row ids.",
    "files": [],
}

for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and p.name != "manifest.json":
        # Avoid hashing huge memmaps to keep this cell quick. Record metadata only.
        item = {
            "path": str(p.relative_to(OUT_DIR)),
            "bytes": int(p.stat().st_size),
        }
        if p.suffix in [".csv", ".json", ".npy"] and p.stat().st_size < 200 * 1024 * 1024:
            item["sha256"] = sha256_file(p)
        else:
            item["sha256"] = None
        manifest["files"].append(item)

with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("wrote", OUT_DIR / "manifest.json")
print("num files:", len(manifest["files"]))

## How to interpret the first run

Check these outputs:

- `basis_correction_error_diagnostic_current.csv`
- `basis_alpha_sweep_proxy_current.csv`
- `basis_proxy_best_summary_current.csv`

Minimum useful signal:

```text
pca_current correlation > 0
best alpha is positive
corrected_top10_overlap does not collapse
```

Useful retrieval-aware signal:

```text
score_error_weighted or top10_boundary_weighted > pca_current
```

If all weighted-SVD bases still lose to PCA after the aligned cache fix, the next step is not more weighting; the next step is pairwise/listwise basis learning on flipped candidates.